# 全屏 A/B 对比（录视频用）

这个 notebook 让 HDMI 在“精确直通”和“自适应处理”之间每 2 秒切换，
用手机对着 HDMI 录 30~60 秒，后期剪成 before/after。

直通 = `MODE_SEL=0 + PARAM=128`（×1.0）；
自适应 = 当前 MLP + 四象限决策 + PARAM。
切换在帧边界生效，不会撕裂。


In [ ]:
import os, sys, time
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, 'lib', 'video_setup.py')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import hard_pair_notebook as hp
from lib import adaptive
print('ROOT =', ROOT)


In [ ]:
ol, regs, vdma0, vdma1 = hp.setup(skip_camera=False, use_manual=True, display=True)


## 先 preview

setup 已启动 HDMI 显示。若屏幕仍黑，先检查显示器输入源/HDMI 线；
不要在 setup 后再调 `vp.start_capture()`，它会把显示通路停掉。

确认当前场景的特征合理，再开始录。


In [ ]:
hp.preview(regs)


## 开始 A/B 切换

默认 10 轮、每边 2 秒；按 Ctrl+C 可提前结束。
录视频时把主体放在画面中央，保持光源/构图不变。


In [ ]:
cycles = 10
seconds = 2.0
try:
    for i in range(cycles):
        regs.set_mode(0)
        regs.set_param(128)
        print('round %d: PASSTHROUGH (mode0, PARAM=128)' % i)
        time.sleep(seconds)
        f = regs.mlp_features()
        q = regs.quads()
        mlp = regs.mlp_output()
        m, p, why = adaptive.decide_spatial(f, q, mlp)
        if m is None:
            m = regs.mode_sel()
            p = adaptive.param_for(f, m)[0]
        regs.set_mode(m)
        regs.set_param(p)
        print('round %d: ADAPTIVE (mode=%d, PARAM=%d) %s' % (i, m, p, why))
        time.sleep(seconds)
except KeyboardInterrupt:
    print('stopped by user')


## 收尾


In [ ]:
hp.cleanup(ol, vdma0, vdma1)
